# Coherence score - does radar coherence help the forest decoder

Part two of the coherence test (written down 2026-10-04 before it was built). Notebook 04a made the coherence of every forest chip. This notebook scores the frozen satellite DINOv3 on forest with and without it.

## What runs

The frozen satellite DINOv3 ViT-L at layer 21 with part two's three channel input, forest, the validation split, the decoder with the seeds 0, 1 and 2, two settings in one session

- as it is, the run step of every saved row
- with coherence, the same features and the coherence as a second channel of the decoder's image skip

The encoder is loaded once and the features are extracted once for both. The probe is the same in both and is not compared.

## The rule, written before the run

The decoder mIoU on val. If the setting with coherence beats the one without by the rule (a gap larger than 0.01 and larger than twice the larger seed SD), coherence helps and a next step, date-matched coherence at 20 m for lakes and snow, is proposed with its cost. Otherwise the coherence question stops here. The IoU of every class is printed next to it (water, built-up and bare ground are where coherence should differ), it does not decide.

## Runtime and time

The L4 GPU, standard RAM is enough (the forest features are about 1 GB). About 10 minutes, an estimate.

## Data and the .env file

The forest dataset as notebook 03 takes it, from its private Hugging Face copy when DATASET_HF_OWNER is set, else from its copy on Drive under MyDrive/datasets, the train and val splits only. The coherence from notebook 04a on Drive. HF_TOKEN reads the gated DINOv3 weights and is never printed.

## Results

Google Drive, folder MyDrive/sar-transfer-distillation/coherence_score/. The files of the run step for both settings and coherence_score.json with the verdict and every row.

## How to run

Run notebook 04a first. Then these cells from top to bottom. A setting whose summary is on Drive is not run again. The last cell flushes Drive and then releases the Colab machine (set END_RUNTIME = False in that cell to keep the runtime), and a cell that fails stops the run before it.


In [ ]:
# Mount Google Drive. The data and the results are there.
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
# <<<SARTRANSFER-SYNC-BEGIN>>>
# Empty sync cell: published notebooks carry no packed code. On your own machine,
# in the repository folder, run   python sync.py   and open the notebook again;
# this cell then holds the package source and unpacks it on the runtime.
raise RuntimeError("The sync cell is empty. On your own machine, in the repository folder, "
                   "run:  python sync.py  and reopen this notebook.")
# <<<SARTRANSFER-SYNC-END>>>

In [ ]:
# Setup. Tokens, model cache, GPU check, folders, the forest dataset on the runtime disk, the coherence of 04a.
import json
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd

from sartransfer import env
from sartransfer.distil import stage0
from sartransfer.domains import hub, prepared
from sartransfer.domains.prepared import load_manifest
import torch

from sartransfer.distil import score
from sartransfer.domains import run as run_step

start = time.perf_counter()
assert torch.cuda.is_available(), "no GPU, pick the L4 runtime and connect again"
DEVICE = "cuda"
MODEL_CACHE = env.model_cache_on_drive()       # before transformers or huggingface_hub load
assert MODEL_CACHE is not None, "Google Drive is not mounted, run the mount cell first"
env.load_env()
HF_TOKEN = env.require("HF_TOKEN", "the gated DINOv3 weights and the private data copies")
assert HF_TOKEN, "no Hugging Face token was given"
DATA_TOKEN = os.environ.get("HF_WRITE_TOKEN") or HF_TOKEN   # the private dataset copies, as notebook 03
DATASET_OWNER = os.environ.get("DATASET_HF_OWNER", "").strip()   # optional, without it the Drive copy is used

# The test, fixed before it was built
DATASET, METRIC = "forty_v1", "mIoU"
ENCODER, MODE = "dinov3-l-sat", "vv_vh_diff"        # part two's run of the frozen satellite DINOv3
LAYER = run_step.PRE_REGISTERED_LAYERS[ENCODER]     # 21, as part two
SPLITS = ("val",)                                   # the validation split
SEEDS = (0, 1, 2)                                   # the decoder seeds of every saved row
NEEDED = ("train", *SPLITS)

RESULTS_FOLDER = "sar-transfer-distillation"     # the Drive folder this notebook writes to, a new name re-runs it from scratch
CONTENT, MYDRIVE = (Path(folder) for folder in stage0.WORK_FOLDERS)
PREP = CONTENT / "sar-transfer-distillation" / "prepared"          # runtime disk, as notebook 03
COH = MYDRIVE / RESULTS_FOLDER / "coherence" / DATASET   # the coherence of notebook 04a, the same folder
OUT = MYDRIVE / RESULTS_FOLDER / "coherence_score"      # the results of this notebook
TIMES = {}                       # seconds of every step of this run, by name


def took(since, name):
    """The time of a cell as text, recorded under `name`."""
    TIMES[name] = time.perf_counter() - since
    return f"cell time {TIMES[name]:.1f} s"


def hide(text):
    """A text without the tokens."""
    out = str(text)
    for secret in (HF_TOKEN, os.environ.get("HF_WRITE_TOKEN")):
        if secret:
            out = out.replace(secret, "<token>")
    return out


def on_disk(name):
    """True when the manifest and the train and val files of a dataset are on the runtime disk."""
    if not (PREP / name / "manifest.json").is_file():
        return False
    return all((PREP / name / file).is_file() for file in prepared.split_files(NEEDED))


if on_disk(DATASET):
    print(f"{DATASET}  on the runtime disk already")
else:
    clock, source = time.perf_counter(), None
    if DATASET_OWNER:
        try:                                       # first the private Hugging Face copy, train and val
            hub.fetch(DATASET, DATASET_OWNER, PREP, token=DATA_TOKEN, splits=NEEDED)
            source = "Hugging Face"
        except Exception as err:
            print(f"{DATASET}  not on Hugging Face ({type(err).__name__}, {hide(err)[:200]})")
    if source is None:                             # then the copy on Drive
        copy = MYDRIVE / "datasets" / hub.repo_name(DATASET)
        assert (copy / "manifest.json").is_file(), f"{DATASET} is neither on Hugging Face nor on Drive under {copy}"
        prepared.copy_prepared(copy, PREP, DATASET, splits=NEEDED)
        source = "Drive"
    print(f"{DATASET}  from {source} in {(time.perf_counter() - clock) / 60:.1f} min")
man = load_manifest(PREP, DATASET)
print(f"{DATASET:<10} splits {man['splits']}, {man['chip_size']} px, units {man['units']}")
record = json.loads((COH / "coherence.json").read_text(encoding="utf-8"))
COHERENCE = {split: np.load(COH / f"{split}_coherence.npy") for split in NEEDED}
for split, arr in COHERENCE.items():
    print(f"coherence {split:<5} {arr.shape}, {record['splits'][split]['chips_without_coherence']} chips without, "
          f"the chips read as the {record['anchor']}")
print(f"GPU     {torch.cuda.get_device_name(0)}")
print(f"results go to   {OUT}")
OUT.mkdir(parents=True, exist_ok=True)
print(took(start, "setup"))


In [ ]:
# Runs. As it is, then with the coherence as a second channel of the decoder's image skip (run_domain's
# extra_skip). The encoder is loaded once and the features are extracted once for both (enc_cache and
# feature_cache), so the two settings differ only in the image skip. A setting whose summary is on Drive is not run
# again.
start = time.perf_counter()
SETTINGS = {"as it is": None, "with coherence": ("coherence", COHERENCE)}
enc_cache, feature_cache = {}, {}
try:
    for setting, extra in SETTINGS.items():
        print(f"\n---- {DATASET}, {setting}")
        run_step.run_domain(PREP, DATASET, ENCODER, OUT, token=HF_TOKEN, layer=LAYER, seeds=SEEDS, mode=MODE,
                            device=DEVICE, splits=SPLITS, enc_cache=enc_cache, feature_cache=feature_cache,
                            extra_skip=extra)
finally:
    enc_cache.clear()
    feature_cache.clear()
    torch.cuda.empty_cache()
print(took(start, "runs"))


In [ ]:
# Verdict. The decoder rows of both settings (seed mean and SD on val), then with coherence against as it is by the
# rule (score.judge), the word by the rule written before the run. The IoU
# of every class is printed next to it. Everything goes to coherence_score.json on Drive.
start = time.perf_counter()
base = score.run_name(DATASET, ENCODER, LAYER, MODE)
RUNS = {"as it is": base, "with coherence": base + "__skip_coherence"}
rows = {}
for setting, run in RUNS.items():
    summary = pd.read_csv(OUT / f"{run}__summary.csv")
    rows[setting] = summary[(summary["split"] == SPLITS[0]) & (summary["head"] == "decoder")]
columns = [c for c in rows["as it is"].columns if c.startswith("IoU_") or c == METRIC]
table = pd.DataFrame({setting: {c: f"{r[c].mean():.4f} ({r[c].std(ddof=1):.4f})" for c in columns}
                      for setting, r in rows.items()})
print("decoder, val, mean (SD) over the seeds")
print(table.to_string())
ref, new = rows["as it is"][METRIC], rows["with coherence"][METRIC]
outcome, gap, needed = score.judge("decoder", new.mean(), new.std(ddof=1), ref.mean(), ref.std(ddof=1))
if outcome == "win":
    verdict = "helps"
    words = (f"COHERENCE HELPS - with coherence beats as it is on the forest mIoU by {gap:.4f} (needed {needed:.4f}). "
             "Option 2 (date-matched coherence at 20 m for lakes and snow) is proposed with its cost")
elif outcome == "missing":
    verdict = "not known"
    words = "NOT KNOWN - a decoder row is missing, see the lines above. Nothing is decided"
else:
    verdict = "no gain"
    words = (f"NO GAIN - with coherence against as it is on the forest mIoU is a {outcome} (gap {gap:.4f}, needed "
             f"{needed:.4f}). The coherence question stops here")
print(f"\n{words}")
REPORT = {"verdict": verdict, "words": words, "outcome": outcome, "gap": gap, "needed": needed,
          "encoder": ENCODER, "layer": LAYER, "mode": MODE, "split": SPLITS[0], "seeds": list(SEEDS),
          "gpu": torch.cuda.get_device_name(0), "coherence": {k: record[k] for k in ("anchor", "placement", "splits")},
          "table": table.to_dict(), "times": TIMES}
(OUT / "coherence_score.json").write_text(json.dumps(REPORT, indent=1, default=float), encoding="utf-8")
print(f"saved {OUT / 'coherence_score.json'}")
print(took(start, "verdict"))


In [ ]:
# Flush Drive and release the runtime. The run is over when this cell prints, Drive then holds
# everything that was written. With END_RUNTIME the Colab machine is released right after, so
# the GPU stops and no compute is spent waiting - google.colab.runtime.unassign asks Colab from
# the machine itself (an HTTP request to the runtime service, read in the colabtools source on
# 2026-10-04), so it works from VS Code as well as from the browser. A cell that fails earlier
# stops the run before this one, so a runtime with an error stays up to be looked at. Set
# END_RUNTIME = False to keep the runtime; run the mount cell again to use Drive afterwards.
from google.colab import drive, runtime

END_RUNTIME = True               # release the machine after the flush, the outputs stay in the notebook

drive.flush_and_unmount()
print("Drive flushed and unmounted, the run is over"
      + (", the runtime is released now" if END_RUNTIME else ", the runtime stays up"), flush=True)
if END_RUNTIME:
    runtime.unassign()
